# Photovoltaic Module Temperature Forecasting

## 01 - Data Preparation

This notebook contains the data preparation steps used to build the dataset for photovoltaic module temperature forecasting.

## 1. Imports

In [1]:
from datetime import time

import numpy as np
import pandas as pd

## 2. Data loading

The original dataset contains solarimetric measurements collected in September and October 2022.  
The raw data files are stored locally and are not redistributed in this repository.

In [2]:
df_september = pd.read_csv(
    "../data/raw/meteo_2022_09_01-30.csv",
    encoding="iso-8859-1"
)

df_october = pd.read_csv(
    "../data/raw/meteo_2022_10_01-31.csv",
    encoding="iso-8859-1"
)

df_total = pd.concat(
    [df_september, df_october],
    ignore_index=True
)

df_total.shape

(965742, 9)

## 3. Initial inspection

In [3]:
# Check dataset dimensions
print("Rows:", df_total.shape[0])
print("Columns:", df_total.shape[1])

Rows: 965742
Columns: 9


In [4]:
# Check column names
df_total.columns

Index(['InstallationUUID', 'Index', 'UpdatedTime', 'IrradianceGlobal',
       'IrradianceDifuse', 'IrradianceGlobalCell', 'IrradiationGlobalTotal',
       'TemperatureAmbient', 'TemperatureModules'],
      dtype='str')

In [5]:
# Convert timestamp column to datetime
df_total["UpdatedTime"] = pd.to_datetime(df_total["UpdatedTime"])

In [6]:
# Check the data collection period
print("Start:", df_total["UpdatedTime"].min())
print("End:", df_total["UpdatedTime"].max())

Start: 2022-09-01 00:01:14
End: 2022-10-31 23:59:19


In [7]:
# Check data types
df_total.dtypes

InstallationUUID                     str
Index                              int64
UpdatedTime               datetime64[us]
IrradianceGlobal                 float64
IrradianceDifuse                 float64
IrradianceGlobalCell             float64
IrradiationGlobalTotal           float64
TemperatureAmbient               float64
TemperatureModules               float64
dtype: object

In [8]:
# Check missing values
df_total.isnull().sum()

InstallationUUID               0
Index                          0
UpdatedTime                    0
IrradianceGlobal          877928
IrradianceDifuse          877928
IrradianceGlobalCell       87814
IrradiationGlobalTotal    877928
TemperatureAmbient             0
TemperatureModules         87814
dtype: int64

## 4. Missing and invalid values


In [9]:
# Fill missing module temperature values
df_total["TemperatureModules"] = df_total["TemperatureModules"].bfill()

In [10]:
df_total["TemperatureModules"].isnull().sum()

np.int64(0)

## 5. Data selection

In [11]:
# Keep only valid measurement records
df_total = df_total.dropna(
    subset=["IrradiationGlobalTotal"]
)

In [12]:
df_total.shape

(87814, 9)

In [13]:
# Select relevant variables
df_total = df_total[
    [
        "UpdatedTime",
        "IrradianceGlobal",
        "TemperatureAmbient",
        "TemperatureModules"
    ]
]

In [14]:
df_total.shape

(87814, 4)

## 6. Time filtering

In [15]:
# Filter records between 06:00 and 18:00
df_total = df_total[
    (df_total["UpdatedTime"].dt.time >= time(6, 0)) &
    (df_total["UpdatedTime"].dt.time <= time(18, 0))
]

In [16]:
df_total.shape

(42653, 4)

In [17]:
print("First time:", df_total["UpdatedTime"].dt.time.min())
print("Last time:", df_total["UpdatedTime"].dt.time.max())

First time: 06:00:13
Last time: 17:59:15


## 7. Final dataset

In [18]:
# Replace invalid zero values with NaN
columns_to_clean = [
    "IrradianceGlobal",
    "TemperatureAmbient",
    "TemperatureModules"
]

df_total[columns_to_clean] = df_total[columns_to_clean].replace(0, np.nan)

In [19]:
df_total.isnull().sum()

UpdatedTime             0
IrradianceGlobal      118
TemperatureAmbient     81
TemperatureModules     84
dtype: int64

In [20]:
# Remove rows containing invalid or missing measurements
df_total = df_total.dropna(subset=columns_to_clean)

In [21]:
df_total.shape

(42532, 4)

In [22]:
df_total.isnull().sum()

UpdatedTime           0
IrradianceGlobal      0
TemperatureAmbient    0
TemperatureModules    0
dtype: int64

## 8. Export for analysis

In [23]:
# Reset index before exporting the final dataset
df_final = df_total.reset_index(drop=True)

# Export the processed dataset
df_final.to_csv(
    "../data/processed/pv_module_temperature_clean.csv",
    index=False
)

print("Dataset exported successfully.")
print("Final shape:", df_final.shape)

Dataset exported successfully.
Final shape: (42532, 4)
